In [0]:
%run ../silver/00_silver_helpers

In [0]:
df= read_table('stores')
display(df)

In [0]:
logger = get_logger("silver_stores")

try:

    logger.info("Starting Silver Stores transformation")

    # ---------------------------------------------------------
    # Data Type Conversion
    # ---------------------------------------------------------

    logger.info("Updating data types of the columns")
    logger.info("Trimming leading/trailing spaces from string columns")

    df = df.select(

        trim(col("store_id").try_cast("string")).alias("store_id"),

        trim(col("store_name").try_cast("string")).alias("store_name"),

        trim(col("city").try_cast("string")).alias("city"),

        trim(col("state").try_cast("string")).alias("state"),

        trim(col("region").try_cast("string")).alias("region"),

        col("_ingestion_timestamp")
            .try_cast("timestamp")
            .alias("_ingestion_timestamp"),

        col("_source_file")
            .try_cast("string")
            .alias("_source_file")
    )

    logger.info(
        "Data type conversion and string trimming completed"
    )

    # ---------------------------------------------------------
    # Duplicate Store IDs
    # ---------------------------------------------------------

    logger.info("Checking for duplicate store_id values")

    before_count = df.count()

    distinct_count = df.dropDuplicates(
        ["store_id"]
    ).count()

    duplicate_count = before_count - distinct_count

    logger.info(
        f"Duplicate store_id records found: {duplicate_count}"
    )

    df = df.dropDuplicates(["store_id"])

    logger.info(
        "Duplicate store_id records removed"
    )

    # ---------------------------------------------------------
    # Null Store IDs
    # ---------------------------------------------------------

    logger.info("Checking for null store_id values")

    before_count = df.count()

    df = df.where(
        col("store_id").isNotNull()
    )

    after_count = df.count()

    records_dropped = before_count - after_count

    logger.info(
        f"Records dropped due to null store_id: {records_dropped}"
    )

    # ---------------------------------------------------------
    # Store Name Cleanup
    # ---------------------------------------------------------

    logger.info(
        "Removing trailing Store IDs from store_name if present"
    )

    df = df.withColumn(
        "store_name",
        split(col("store_name"), " ")[0]
    )

    logger.info(
        "Store IDs removed from store_name"
    )
    df.printSchema()

    display(df)
    
    # ---------------------------------------------------------
    # Schema
    # ---------------------------------------------------------

    logger.info(
        f"Creating schema if it does not exist: "
        f"{catalog_name}.{schema_name}"
    )

    spark.sql(
        f"""
        CREATE SCHEMA IF NOT EXISTS
        {catalog_name}.{schema_name}
        """
    )

    logger.info(
        f"Schema ready: {catalog_name}.{schema_name}"
    )

    # ---------------------------------------------------------
    # Save
    # ---------------------------------------------------------

    logger.info("Saving Silver Stores table")

    save_table(
        df,
        "stores_clean"
    )

    logger.info(
        "Silver Stores table saved successfully"
    )

    logger.info(
        "Silver Stores transformation completed successfully"
    )


except Exception:

    logger.exception(
        "Silver Stores transformation failed"
    )

    raise